### Load Data & Build BM25 Index

In [5]:
import pandas as pd
import time
import urllib3
from elasticsearch import Elasticsearch, helpers

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Connecting to Elasticsearch...")
es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", "6zoRJw9FqjrVa2hO=PP_"),
    ca_certs="http_ca.crt"
)
print("Connected successfully:", es.info().body['cluster_name'])

print("\nLoading preprocessed dataset...")
df = pd.read_csv('../../data_resource/preprocessed_recipes.csv')
df = df.fillna('')
print(f"Dataset loaded: {len(df)} rows.")

index_name = "se481_recipes"

print("\nPreparing Elasticsearch Index...")
start_time = time.time()

if es.indices.exists(index=index_name):
    print(f"Deleting old index: {index_name}")
    es.indices.delete(index=index_name)

mapping = {
    "mappings": {
        "properties": {
            "RecipeId": {"type": "integer"},
            "Name": {"type": "text"},
            "RecipeCategory": {"type": "keyword"},
            "RecipeIngredientParts": {"type": "text"},
            "RecipeInstructions": {"type": "text"},
            "Image_URL": {"type": "keyword"},
            "clean_text": {"type": "text"}
        }
    }
}
es.indices.create(index=index_name, body=mapping)
print(f"Created new index: '{index_name}'")

def generate_actions(dataframe):
    for _, row in dataframe.iterrows():
        yield {
            "_index": index_name,
            "_source": {
                "RecipeId": row["RecipeId"],
                "Name": row["Name"],
                "RecipeCategory": row.get("RecipeCategory", "Unknown"),
                "RecipeIngredientParts": row["RecipeIngredientParts"],
                "RecipeInstructions": row["RecipeInstructions"],
                "Image_URL": row["Image_URL"],
                "clean_text": row["clean_text"]
            }
        }

print("\nStarting Bulk Ingestion into Elasticsearch (This might take 1-3 minutes)...")
ingest_start = time.time()

success, failed = helpers.bulk(es, generate_actions(df), chunk_size=2000)

print(f"Data Ingestion completed in {time.time() - ingest_start:.2f} seconds.")
print(f"Successfully indexed: {success} recipes into '{index_name}'.")

Connecting to Elasticsearch...
Connected successfully: docker-cluster

Loading preprocessed dataset...
Dataset loaded: 522517 rows.

Preparing Elasticsearch Index...
Created new index: 'se481_recipes'

Starting Bulk Ingestion into Elasticsearch (This might take 1-3 minutes)...
Data Ingestion completed in 62.96 seconds.
Successfully indexed: 522517 recipes into 'se481_recipes'.


### Test Search & Save Model

In [6]:
import time

query = "spicy chicken noodle"
print(f"Query: '{query}'")
print("Retrieving top 5 results from Elasticsearch...")

start_time = time.time()

response = es.search(
    index="se481_recipes",
    body={
        "query": {
            "match": {
                "clean_text": query  # ค้นหาจากคอลัมน์ที่เรา clean ไว้
            }
        },
        "size": 5  # เอาแค่ 5 อันดับแรก
    }
)

search_time = (time.time() - start_time) * 1000  # แปลงเป็นมิลลิวินาที

print(f"\n--- Top 5 Results (Found in {search_time:.2f} ms) ---")

hits = response["hits"]["hits"]
for i, hit in enumerate(hits, 1):
    score = hit["_score"] # คะแนนความเหมือน (BM25 Score)
    name = hit["_source"]["Name"] # ชื่อเมนู
    print(f"Rank {i} [Score: {score:.4f}]: {name}")

print("--------------------------------------")
print("Search test successful!")

Query: 'spicy chicken noodle'
Retrieving top 5 results from Elasticsearch...

--- Top 5 Results (Found in 1903.99 ms) ---
Rank 1 [Score: 16.9321]: Spicy Shoyu Ramen Noodle
Rank 2 [Score: 14.7398]: Spicy Asian Chicken Noodle Salad
Rank 3 [Score: 13.8416]: Spicy Asian Chicken and Noodle Salad
Rank 4 [Score: 13.8416]: Spicy Noodle Salad With Chicken and Mint
Rank 5 [Score: 13.2428]: Spicy Chicken &amp; Bok Choy Noodle Bowl
--------------------------------------
Search test successful!
